# CAMELS — Make Input Patches (v3, slim)

Pipeline minima per generare le **patch di input** per il training U-Net.

- **X**: canale unico di densita` SPH (il canale h_sph e` stato eliminato: h ∝ ρ^(-1/3), Spearman ρ=+0.81 → nessuna informazione nuova — decisione prof. Benitez-Llambay, 29/09).
- **Y**: label dal **catalogo FoF/SubFind** (`groups_090.hdf5`), non derivate dai picchi del campo.
- Output in `C:\Users\kekko\Downloads\Python\TESI\CAMELS_data\input_data\`.
- Niente diagnostica/EDA: solo rendering → normalizzazione → patch → salvataggio.

In [ ]:
# ============ CONFIG ============
DATA_DIR     = r"C:\Users\kekko\Downloads\Python\TESI\CAMELS_data"
SNAP_NAME    = "snapshot_090.hdf5"
CATALOG_NAME = "groups_090.hdf5"
OUT_FOLDER   = "input_data"                            # <-- cartella di output richiesta

# SPH / rendering (invariati rispetto a v2.3)
N_NEIGHBORS_H   = 32       # k-nearest per estimate_h
TARGET_CELLS    = 4        # target_cells_per_h nel render
R_MAX_LEVEL     = 10       # -> IMG_SIZE = 2**10 = 1024
IMG_SIZE        = 2 ** R_MAX_LEVEL
N_SLICES        = 25       # fette in z, Δz = 1 Mpc/h

# Label da catalogo
N_MIN_PARTICLES = 100                          # FoF default
MASS_CUT_MSUNH  = None                         # riempito dopo il caricamento = N_MIN * m_DM
HALO_WEIGHTING  = "center"                     # "center" (v1) | "fraction" (peso volumetrico nella fetta)
MIN_RADIUS_PX   = 1.5                          # floor sul raggio disegnato

# Patch
PATCH_SIZE      = 256                          # 1024 / 4 → griglia 4x4, non sovrapposte

SEED = 42


## 1. Dipendenze, snapshot, geometria delle fette

In [ ]:
import os, time
import numpy as np
import h5py
import hdf5plugin                      # necessario per la compressione CAMELS
from scipy.ndimage import gaussian_filter, maximum_filter
from sphviewer2.tools import estimate_h
from sphviewer2.core import Settings
from sphviewer2.render import sph_render

SNAP_PATH    = os.path.join(DATA_DIR, SNAP_NAME)
CATALOG_PATH = os.path.join(DATA_DIR, CATALOG_NAME)
OUT_DIR      = os.path.join(DATA_DIR, OUT_FOLDER)
os.makedirs(OUT_DIR, exist_ok=True)

t0 = time.time()
with h5py.File(SNAP_PATH, "r") as f:
    pos       = np.asarray(f["PartType1/Coordinates"], dtype=np.float64)   # ckpc/h
    mass_tab  = np.asarray(f["Header/MassTable"], dtype=np.float64)
    box_size  = float(np.asarray(f["Header/BoxSize"]).ravel()[0])           # 25000 ckpc/h
print(f"particelle: {pos.shape[0]:,} | BoxSize: {box_size} ckpc/h | m_DM: {mass_tab[1]:.4e} Msun/h")
print(f"caricamento: {time.time()-t0:.1f}s")

M_DM    = float(mass_tab[1])
LBOX_MPC = box_size / 1000.0                       # 25 Mpc/h
MASS_CUT_MSUNH = N_MIN_PARTICLES * M_DM            # 2.6004e9 Msun/h
PIXEL_CKPC     = box_size / IMG_SIZE               # 24.41 ckpc/h
SLICE_DZ_MPC   = LBOX_MPC / N_SLICES               # 1.0 Mpc/h


## 2. Smoothing lengths (cache riutilizzabile)

Il kernel SPH di `sphviewer2` ha **supporto compatto** (cubic spline piecewise), non gaussiano: le code muoiono a 1–2 lunghezze di smoothing, ed e` questo che rende il render veloce.

In [ ]:
# il nome della cache dipende dal dataset (evita collisioni quando aggiungerai altre simulazioni CAMELS)
import hashlib
_ds_tag = hashlib.md5(os.path.basename(SNAP_PATH).encode()).hexdigest()[:8]
H_CACHE = os.path.join(DATA_DIR, f"smoothing_lengths_{_ds_tag}_k{N_NEIGHBORS_H}.npy")
t0 = time.time()
if os.path.exists(H_CACHE):
    h_sph = np.load(H_CACHE)
    print(f"h_sph da cache: {H_CACHE}")
else:
    print("calcolo h_sph (estimate_h, ~1-3 min)...")
    h_sph = estimate_h(pos.astype(np.float32), k=N_NEIGHBORS_H, box=[box_size]*3)
    h_sph = np.asarray(h_sph, dtype=np.float64)
    good = np.isfinite(h_sph) & (h_sph > 0)
    pos, h_sph = pos[good], h_sph[good]
    np.save(H_CACHE, h_sph)
print(f"particelle usate: {pos.shape[0]:,} | tempo: {time.time()-t0:.1f}s")


## 3. Rendering SPH delle 25 fette (1 canale, log10)

In [ ]:
def render_slice_density(pos, h, z_lo, z_hi, settings):
    """Proiezione SPH delle particelle con z in [z_lo, z_hi). Ritorna immagine grezza (IMG_SIZE, IMG_SIZE)."""
    sel = (pos[:, 2] >= z_lo) & (pos[:, 2] < z_hi)
    x = np.column_stack([pos[sel, :2], np.zeros(sel.sum())])
    img = sph_render(x, h[sel], settings=settings, periodic=True)
    return np.asarray(img, dtype=np.float32)

settings = Settings(
    dim=2, level=R_MAX_LEVEL,
    target_cells_per_h=TARGET_CELLS,
    box_size=[box_size, box_size],
)

t0 = time.time()
density_slices = np.zeros((N_SLICES, IMG_SIZE, IMG_SIZE), dtype=np.float32)
for i in range(N_SLICES):
    z_lo = i * SLICE_DZ_MPC
    z_hi = z_lo + SLICE_DZ_MPC
    density_slices[i] = render_slice_density(pos, h_sph, z_lo, z_hi, settings)
print(f"render {N_SLICES} fette completato in {time.time()-t0:.1f}s")

np.save(os.path.join(OUT_DIR, "slices_density_raw.npy"), density_slices)


## 4. Normalizzazione robusta (solo CH1)

log10 della densita`, poi z-score con mediana e sigma robusto σ=(P84−P16)/2 calcolati su **tutte** le fette.

In [ ]:
with np.errstate(divide="ignore"):
    log_dens = np.log10(density_slices, where=density_slices > 0,
                        out=np.full_like(density_slices, np.nan))
finite = np.isfinite(log_dens)
rng_sub = np.random.default_rng(SEED)                       # percentile su un campione di ~262k px
samp_idx = rng_sub.choice(finite.size, size=min(262144, int(finite.sum())), replace=False)
p16, med_d, p84 = np.percentile(log_dens.reshape(-1)[samp_idx], [16, 50, 84])
sig_d = (p84 - p16) / 2.0
dens_norm = (log_dens - med_d) / sig_d
dens_norm[~np.isfinite(dens_norm)] = -3.0   # pixel vuoti -> valore basso costante
norm_stats = np.array([[med_d, sig_d]], dtype=np.float32)   # shape (1,2): solo CH1
print(f"MED_D={med_d:.4f}  SIG_D={sig_d:.4f}")


## 5. Catalogo FoF/SubFind → coordinate aloni

In [ ]:
with h5py.File(CATALOG_PATH, "r") as f:
    candidates = [k for k in ("Group", "GROUPOBJECT", "CatalogObject") if k in f]
    grp_name = candidates[0] if candidates else list(f.keys())[0]
    g = f[grp_name]
    def get(*names):
        for n in names:
            if n in g:
                return np.asarray(g[n], dtype=np.float64)
        raise KeyError(names)
    gpos   = get("GroupPos", "gropos")                 # ckpc/h
    gmass  = get("GroupMass", "gomass")                # 1e10 Msun/h
    gr200  = get("Group_R_Crit200", "GOR_Crit200")     # ckpc/h
print(f"aloni totali: {len(gmass)}")

keep = (gmass * 1e10 >= MASS_CUT_MSUNH) & (gr200 > 0)
gpos, gmass, gr200 = gpos[keep], gmass[keep], gr200[keep]
halo_xyz_mpc  = gpos / 1000.0            # Mpc/h
halo_mass     = gmass * 1e10             # Msun/h
halo_r200_ckpc = gr200                   # ckpc/h
print(f"aloni sopra mass cut ({MASS_CUT_MSUNH:.3e} Msun/h): {len(halo_mass)}")


## 6. Disegno label (periodico!)

`make_halo_labels` disegna la maschera binaria di ogni alone sulla fetta usando la **distanza periodica minima** (il render e` `periodic=True`; la vecchia `build_halo_mask` di v1 non era periodica → bug da non reintrodurre). Con `HALO_WEIGHTING="fraction"` l`etichetta e` pesata dalla frazione volumetrica della sfera R200c dentro la fetta (calotta sferica); con `"center"` ogni alone va solo nella fetta del suo centro.

In [ ]:
def _slice_weight(z_center, r_mpc, z_lo, z_hi):
    """Peso dell'alone nella fetta [z_lo, z_hi)."""
    if HALO_WEIGHTING == "center":
        return 1.0 if (z_lo <= z_center < z_hi) else 0.0
    dz = z_hi - z_lo
    if r_mpc <= 0:
        return 0.0
    lo = max(z_center - r_mpc, z_lo)
    hi = min(z_center + r_mpc, z_hi)
    if hi <= lo:
        return 0.0
    chord = hi - lo
    # volume calotta sferica: V = pi*h^2*(3R-h)/3, h = altezza della porzione di sfera nella fetta
    # frazione = somma dei volumi delle due calotte ritagliate dalle facce della fetta
    def cap_vol(h):
        return np.pi * h**2 * (3*r_mpc - h) / 3.0
    v_full = 4.0/3.0 * np.pi * r_mpc**3
    d_top = z_center + r_mpc - z_hi      # quanto sporge sopra
    d_bot = z_lo - (z_center - r_mpc)    # quanto sporge sotto
    v = v_full
    if d_top > 0: v -= cap_vol(min(d_top, 2*r_mpc))
    if d_bot > 0: v -= cap_vol(min(d_bot, 2*r_mpc))
    return float(np.clip(v / v_full, 0.0, 1.0))

def make_halo_labels(centers_xy_px, radii_px, weights, img_size=IMG_SIZE):
    """Maschera binaria (uint8) con geometria periodica. centers/radi in px."""
    frac = any(0 < w < 1 for w in weights)
    y = np.zeros((img_size, img_size), dtype=np.float32 if frac else np.uint8)
    for (cx, cy), r, w in zip(centers_xy_px, radii_px, weights):
        if w <= 0:
            continue
        r = max(int(round(float(r))), int(np.ceil(MIN_RADIUS_PX)))
        cx_i, cy_i = int(np.floor(cx)) % img_size, int(np.floor(cy)) % img_size
        ys = np.arange(cy_i - r, cy_i + r + 1)
        xs = np.arange(cx_i - r, cx_i + r + 1)
        dy = np.abs(ys - cy_i); dy = np.minimum(dy, img_size - dy)
        dx = np.abs(xs - cx_i); dx = np.minimum(dx, img_size - dx)
        disk = ((dx[:, None]**2 + dy[None, :]**2) <= r**2).astype(np.float32) * min(w, 1.0)
        yy = ys % img_size; xx = xs % img_size
        y[np.ix_(yy, xx)] = np.maximum(y[np.ix_(yy, xx)], disk.astype(y.dtype))
    return y


## 7. Label per fetta

In [ ]:
t0 = time.time()
label_slices = np.zeros((N_SLICES, IMG_SIZE, IMG_SIZE),
                      dtype=np.float32 if HALO_WEIGHTING == "fraction" else np.uint8)
px_per_mpc = IMG_SIZE / LBOX_MPC
for i in range(N_SLICES):
    z_lo = i * SLICE_DZ_MPC
    z_hi = z_lo + SLICE_DZ_MPC
    centers, radii, weights = [], [], []
    for xyz, r_ckpc in zip(halo_xyz_mpc, halo_r200_ckpc):
        w = _slice_weight(xyz[2], r_ckpc / 1000.0, z_lo, z_hi)
        if w > 0:
            centers.append((xyz[0] * px_per_mpc, xyz[1] * px_per_mpc))
            radii.append(r_ckpc / 1000.0 * px_per_mpc)
            weights.append(w)
    if centers:
        label_slices[i] = make_halo_labels(centers, radii, weights)
if HALO_WEIGHTING == "fraction":
    label_slices = (label_slices >= 0.5).astype(np.uint8)   # pesi frazionari -> maschera {0,1} al 50%
print(f"label create in {time.time()-t0:.1f}s | px con alone: {label_slices.sum():,} "
      f"({100*label_slices.mean():.2f}% dell'area)")


## 8. Estrazione patch 4×4 non sovrapposte e salvataggio

Tutte le patch vengono tenute, incluse quelle completamente negative (voluto: addestrano la rete a dire "qui non c'e` niente").

In [ ]:
n_grid = IMG_SIZE // PATCH_SIZE
X = np.zeros((N_SLICES * n_grid**2, 1, PATCH_SIZE, PATCH_SIZE), dtype=np.float32)
Y = np.zeros((N_SLICES * n_grid**2, 1, PATCH_SIZE, PATCH_SIZE), dtype=np.float32)
k = 0
for i in range(N_SLICES):
    for gy in range(n_grid):
        for gx in range(n_grid):
            sy, sx = gy*PATCH_SIZE, gx*PATCH_SIZE
            X[k, 0] = dens_norm[i, sy:sy+PATCH_SIZE, sx:sx+PATCH_SIZE]
            Y[k, 0] = label_slices[i, sy:sy+PATCH_SIZE, sx:sx+PATCH_SIZE]
            k += 1
n_pos = (Y[:, 0].sum(axis=(1, 2)) > 0).sum()
print(f"X {X.shape}, Y {Y.shape} | patch positive: {n_pos}/{len(X)} ({100*n_pos/len(X):.1f}%)")

np.save(os.path.join(OUT_DIR, "X_patches.npy"), X)
np.save(os.path.join(OUT_DIR, "Y_mask_patches.npy"), Y)
np.save(os.path.join(OUT_DIR, "norm_stats.npy"), norm_stats)

with open(os.path.join(OUT_DIR, "dataset_info.txt"), "w") as fh:
    fh.write(
        "CAMELS input_data (slim v3)\n"
        f"snapshot: {SNAP_PATH}\ncatalogo: {CATALOG_PATH}\n"
        f"N_SLICES={N_SLICES} IMG_SIZE={IMG_SIZE} PATCH_SIZE={PATCH_SIZE}\n"
        f"canali X: 1 (log10 densita SPH, z-score robusto MED_D={med_d:.4f} SIG_D={sig_d:.4f})\n"
        f"Y: maschera binaria da catalogo FoF/SubFind (Group_R_Crit200), weighting={HALO_WEIGHTING}\n"
        f"MASS_CUT={MASS_CUT_MSUNH:.4e} Msun/h (N_min={N_MIN_PARTICLES} x m_DM={M_DM:.4e})\n"
        f"MIN_RADIUS_PX={MIN_RADIUS_PX}\n"
        f"patch: {len(X)} totali, {n_pos} positive\n"
    )
print("salvato in:", OUT_DIR)
import os as _os
for fn in sorted(_os.listdir(OUT_DIR)):
    p = _os.path.join(OUT_DIR, fn)
    print(f"  {fn:32s} {_os.path.getsize(p)/1e6:8.1f} MB")
